In [51]:
import polars as pl
from depersonal_lib.core.transformations import generalize, suppress_rare_categories, drop_column
from depersonal_lib.core.privacy_models import KAnonymity
from depersonal_lib.core.transformations import EnteringIdentifiers
from depersonal_lib.core.transformations import ShuffleTransformer
from depersonal_lib.core.transformations import DecomposeTransformer
from depersonal_lib.core.transformations import DifferentialPrivacy

## Пример использования обобщения

In [9]:
# Создаем тестовые данные
df = pl.DataFrame({
    "id": [1, 2, 3, 4],
    "age": [23, 29, 45, 51]
})
df

id,age
i64,i64
1,23
2,29
3,45
4,51


In [10]:
# Обобщение возраста (интервалы: 0-30, 30-60, 60-100)
df = generalize(df, 'age', bins=[0, 30, 60, 100])
df

id,age
i64,str
1,"""1-30"""
2,"""1-30"""
3,"""31-60"""
4,"""31-60"""


## Пример подавления редких категорий

In [11]:
# Создаем тестовые данные
df = pl.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "occupation": ['teacher','engineer','teacher','pilot','AI researcher','engineer']
})
df

id,occupation
i64,str
1,"""teacher"""
2,"""engineer"""
3,"""teacher"""
4,"""pilot"""
5,"""AI researcher"""
6,"""engineer"""


In [12]:
# Подавление редких категорий в колонке 'occupation'
df = suppress_rare_categories(df, column="occupation", threshold=0.3)
df

id,occupation
i64,str
1,"""teacher"""
2,"""engineer"""
3,"""teacher"""
4,"""Other"""
5,"""Other"""
6,"""engineer"""


## Пример исключения столбцов

In [13]:
# Создаем тестовые данные
df = pl.DataFrame({
    "id": [1, 2],
    "name": ['Ivan Petrov','Anna Smirnova'],
    "age": [32, 27],
    "diagnosis": ['Hypertension', 'Asthma']
})
df

id,name,age,diagnosis
i64,str,i64,str
1,"""Ivan Petrov""",32,"""Hypertension"""
2,"""Anna Smirnova""",27,"""Asthma"""


In [14]:
data_without_column = drop_column(df, "name")
data_without_column

id,age,diagnosis
i64,i64,str
1,32,"""Hypertension"""
2,27,"""Asthma"""


## Пример введение идентификаторов

In [2]:
# Создаем тестовые данные
df = pl.DataFrame({
    "id": [1, 2, 3],
    "email": ['alice@example.com','bob@example.org','carol@example.net']
})
df

id,email
i64,str
1,"""alice@example.com"""
2,"""bob@example.org"""
3,"""carol@example.net"""


In [3]:
# Инициализируем преобразователь для колонки "email"
transformer = EnteringIdentifiers("email")

In [17]:
# Применяем трансформацию
anonymized_data = transformer.transform(df)
print("Обезличенные данные:")
anonymized_data

Обезличенные данные:


id,id_d325f7
i64,str
1,"""a357c4cbdfa4423b8da9bac253e847…"
2,"""cdab767c9628469eba8243e911a0d7…"
3,"""44865b32f1384acb9768861dbbbf11…"


In [19]:
# Сохраняем таблицу соответствий
transformer.save_mapping("mapping.csv", format="csv")

In [20]:
mapping_tabel = pl.read_csv('mapping.csv')
mapping_tabel

identifier,email,target_column
str,str,str
"""44865b32f1384acb9768861dbbbf11…","""carol@example.net""","""id_d325f7"""
"""a357c4cbdfa4423b8da9bac253e847…","""alice@example.com""","""id_d325f7"""
"""cdab767c9628469eba8243e911a0d7…","""bob@example.org""","""id_d325f7"""


In [21]:
# Восстановление
restored_df = transformer.restore(
    df=anonymized_data,
    column=mapping_tabel['target_column'][0],  # Имя обезличенной колонки id_***
    mapping_path="mapping.csv"
)
restored_df

id,email
i64,str
1,"""alice@example.com"""
2,"""bob@example.org"""
3,"""carol@example.net"""


## Пример перестановки значений

In [22]:
# Создаем тестовые данные
df = pl.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "age": [25, 30, 45, 22, 37]
})
df

id,age
i64,i64
1,25
2,30
3,45
4,22
5,37


In [24]:
shuffler_transform = ShuffleTransformer("age")

In [39]:
shuffled_df = shuffler_transform.shuffle(df, seed=48)
shuffled_df

id,age
i64,i64
1,37
2,22
3,45
4,30
5,25


In [40]:
# Восстановление
restored_df = shuffler_transform.unshuffle(shuffled_df)
restored_df

id,age
i64,i64
1,25
2,30
3,45
4,22
5,37


## Пример декомпозиции данных

In [42]:
# Создаем тестовые данные
df = pl.DataFrame({
    "name": ['Ivan', 'Maria', 'Sergey'],
    "age": [32, 27, 45],
    "email": ['ivan@mail.com', 'maria@example.org', 'sergey@test.net']
})
df

name,age,email
str,i64,str
"""Ivan""",32,"""ivan@mail.com"""
"""Maria""",27,"""maria@example.org"""
"""Sergey""",45,"""sergey@test.net"""


In [43]:
# Инициализация преобразователя
decomp_transformer = DecomposeTransformer(
    part1_columns=["name", "age"],
    part2_columns=["email"]
)

In [44]:
part1, part2 = decomp_transformer.decompose(df)
part1

name,age,id_part1_64fa52
str,i64,str
"""Ivan""",32,"""2841869f56f34a8abd81c6244a6328…"
"""Maria""",27,"""96c1dfd565944d8fa21367117722d9…"
"""Sergey""",45,"""ba8dbe4adf08449ebe36548579682d…"


In [45]:
part2

email,id_part2_6a5c02
str,str
"""ivan@mail.com""","""8e324eee6ad34baa9819989d514049…"
"""maria@example.org""","""aff247aa5d254eb9bf030bd89bde0d…"
"""sergey@test.net""","""3f6f162274414206a91c37c3641a3a…"


In [46]:
# Сохранение таблицы соответствий
decomp_transformer.save_mapping("decompose_mapping.csv", format="csv")

In [47]:
# Загрузка таблицы соответствий
loaded_mapping = DecomposeTransformer.load_mapping("decompose_mapping.csv", format="csv")

In [48]:
loaded_mapping

id_part1_64fa52,id_part2_6a5c02
str,str
"""2841869f56f34a8abd81c6244a6328…","""8e324eee6ad34baa9819989d514049…"
"""96c1dfd565944d8fa21367117722d9…","""aff247aa5d254eb9bf030bd89bde0d…"
"""ba8dbe4adf08449ebe36548579682d…","""3f6f162274414206a91c37c3641a3a…"


In [49]:
# Восстановление данных
restored_df = DecomposeTransformer.restore(part1, part2, loaded_mapping)
restored_df

name,age,email
str,i64,str
"""Ivan""",32,"""ivan@mail.com"""
"""Maria""",27,"""maria@example.org"""
"""Sergey""",45,"""sergey@test.net"""


In [50]:
# Оригинальные данные
df

name,age,email
str,i64,str
"""Ivan""",32,"""ivan@mail.com"""
"""Maria""",27,"""maria@example.org"""
"""Sergey""",45,"""sergey@test.net"""


## Пример добавления шума

In [52]:
# Создаем тестовые данные
df = pl.DataFrame({
    "id": [1, 2, 3],
    "salary": [50000, 60000, 55000]
})
df

id,salary
i64,i64
1,50000
2,60000
3,55000


In [93]:
# Инициализация
dp_laplace = DifferentialPrivacy(
    epsilon=0.5,
    sensitivity={'salary': 10000.0}
)
dp_gauss = DifferentialPrivacy(
    epsilon=0.5,
    delta=1e-5,
    sensitivity={'salary': 10000.0}
)

In [120]:
# Зашумление методом Лапласа
df_laplace = dp_laplace.add_laplace_noise(df, ['salary'])
df_laplace

id,salary
i64,i64
1,66963
2,94309
3,68203


In [179]:
# Зашумление методом Гаусса
df_gaussian = dp_gauss.add_gaussian_noise(df, ['salary'])
df_gaussian

id,salary
i64,i64
1,18082
2,52674
3,17285
